# Satellite Visual Search — 50k-chip demo dataset builder

Builds a **50,000-chip demo dataset** from the xView satellite imagery (Kaggle) that matches what the backend expects:

| Output | Content |
|---|---|
| `embeddings.npy` | `(N, 384)` float32, **raw (unnormalized) DINOv2-small CLS** — same as `backend/services/ml_service.py` |
| `chips.parquet` / `chips.csv` | `idx, chip_id, lat, lon, scale, source, tile_id, x_offset, y_offset, chip_path, ground_truth_class` (same schema as the SQLite `chips` table) |
| `thumbs/*.webp` + `thumbs.zip` | one small thumbnail per chip |

Optional last cells load everything into **Supabase (Postgres + pgvector + Storage)**.

**Runs best on Kaggle** (attach the xView dataset, enable GPU T4). It also runs locally via `kagglehub`.

In [ ]:
# ══ PLACEHOLDERS — fill these in before "Run all" ═════════════════════════════
# 1) Dataset ------------------------------------------------------------------
#    On Kaggle: click "+ Add Input" and attach the xView dataset; leave KAGGLE_SLUG empty.
#    Off Kaggle: slug is the part of the dataset URL after kaggle.com/datasets/, e.g. "owner/xview-dataset".
KAGGLE_SLUG   = ""
LOCAL_XVIEW   = ""            # off Kaggle only: folder that already holds the xView *.tif files

# 2) Supabase (optional) -------------------------------------------------------
#    Turn these on only after the dataset cells work and your Supabase project exists.
LOAD_TO_SUPABASE = False      # create table + load vectors/metadata + HNSW index
UPLOAD_THUMBS    = False      # upload thumbnails to Supabase Storage
#    NAMES of your Kaggle Secrets (Notebook > Add-ons > Secrets) — not the values themselves.
SECRET_DB_URL      = "DATABASE_URL"          # Supabase > Project Settings > Database > Connection string > Session pooler
SECRET_SB_URL      = "SUPABASE_URL"          # Supabase > Project Settings > API > Project URL
SECRET_SB_KEY      = "SUPABASE_SERVICE_KEY"  # Supabase > Project Settings > API > service_role key (keep private!)
STORAGE_BUCKET     = "thumbs"                # create a PUBLIC bucket with this name in Supabase > Storage

# 3) Dataset size / chips (defaults match the existing backend index) ------------
N_CHIPS       = 50_000
CHIP_SIZE     = 128           # matches dino_128.index
STRIDE        = 96
SEED          = 42
THUMB_SIZE    = 128
BATCH_SIZE    = 128
OUT_DIR       = "demo_dataset"
SOURCE        = "xview"

In [ ]:
import sys, subprocess
def pip(*pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
try:
    import rasterio, transformers, pyarrow, psycopg  # noqa
except ImportError:
    pip("rasterio", "transformers", "pyarrow", "pandas", "pillow", "tqdm", "kagglehub", "psycopg[binary]", "pgvector", "supabase")

In [ ]:
import os, io, json, glob, math, zipfile, random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm
import rasterio
from rasterio.warp import transform as rio_transform
from transformers import AutoImageProcessor, AutoModel

Image.MAX_IMAGE_PIXELS = None
rng = np.random.default_rng(SEED); random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path(OUT_DIR); (OUT / "thumbs").mkdir(parents=True, exist_ok=True)
print("device:", DEVICE)

## 1. Locate the xView data

In [ ]:
def find_root():
    if LOCAL_XVIEW: return Path(LOCAL_XVIEW)
    if Path("/kaggle/input").exists():
        return Path("/kaggle/input")
    import kagglehub
    assert KAGGLE_SLUG, "Set KAGGLE_SLUG (or LOCAL_XVIEW) in the config cell"
    return Path(kagglehub.dataset_download(KAGGLE_SLUG))

ROOT = find_root()
tifs = sorted(glob.glob(str(ROOT / "**" / "*.tif"), recursive=True))
# keep training images only if the dataset also ships val images
train_tifs = [t for t in tifs if "val" not in Path(t).parent.name.lower()] or tifs
geojson = next(iter(glob.glob(str(ROOT / "**" / "xView_train.geojson"), recursive=True)), None)
print(f"{len(train_tifs)} tifs | labels: {geojson}")
assert train_tifs, "No .tif files found — check the dataset slug / path"

## 2. Plan which chips to extract
Even coverage across all tiles, using the same sliding-window grid as the backend (`stride = 96`). Only full-size chips are kept.

In [ ]:
plan = []   # (tif_path, [(x, y), ...])
per_tile = math.ceil(N_CHIPS / len(train_tifs))
for t in tqdm(train_tifs, desc="grid"):
    with Image.open(t) as im: W, H = im.size
    grid = [(x, y) for y in range(0, H - CHIP_SIZE + 1, STRIDE) for x in range(0, W - CHIP_SIZE + 1, STRIDE)]
    if not grid: continue
    k = min(per_tile, len(grid))
    pick = rng.choice(len(grid), size=k, replace=False)
    plan.append((t, [grid[i] for i in sorted(pick)]))

# trim to exactly N_CHIPS (drop random extras) so the dataset size is predictable
flat = [(ti, c) for ti, (_, cs) in enumerate(plan) for c in cs]
if len(flat) > N_CHIPS:
    drop = set(rng.choice(len(flat), size=len(flat) - N_CHIPS, replace=False).tolist())
    keep = [f for i, f in enumerate(flat) if i not in drop]
    plan = [(p, []) for p, _ in plan]
    for ti, c in keep: plan[ti][1].append(c)
total = sum(len(c) for _, c in plan)
print("tiles:", len(plan), "| chips planned:", total)

## 3. Ground-truth labels (optional)
Each chip gets the most frequent xView class (`type_id`) among label boxes whose centre falls inside it, else `None`.

In [ ]:
labels = {}   # image file name -> array [cx, cy, type_id]
if geojson:
    with open(geojson) as f: gj = json.load(f)
    tmp = {}
    for ft in tqdm(gj["features"], desc="labels"):
        p = ft["properties"]
        x1, y1, x2, y2 = map(float, p["bounds_imcoords"].split(","))
        tmp.setdefault(p["image_id"], []).append(((x1 + x2) / 2, (y1 + y2) / 2, int(p["type_id"])))
    labels = {k: np.array(v) for k, v in tmp.items()}
    del gj
def chip_class(name, x, y):
    a = labels.get(name)
    if a is None: return None
    m = (a[:, 0] >= x) & (a[:, 0] < x + CHIP_SIZE) & (a[:, 1] >= y) & (a[:, 1] < y + CHIP_SIZE)
    return str(Counter(a[m, 2].astype(int).tolist()).most_common(1)[0][0]) if m.any() else None
print("tiles with labels:", len(labels))

## 4. Extract chips → DINOv2-small embeddings → thumbnails
Same model and preprocessing as `backend/services/ml_service.py` (CLS token, **not** normalized).

In [ ]:
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
model = AutoModel.from_pretrained("facebook/dinov2-small").to(DEVICE).eval()
if DEVICE == "cuda": model.half()

def embed(chips):
    inp = processor(images=chips, return_tensors="pt")
    pv = inp["pixel_values"].to(DEVICE)
    if DEVICE == "cuda": pv = pv.half()
    with torch.no_grad():
        return model(pixel_values=pv).last_hidden_state[:, 0, :].float().cpu().numpy()

def latlon(src, cx, cy):
    xs, ys = src.xy(cy, cx)           # pixel (row, col) -> CRS coords
    if src.crs and src.crs.to_epsg() != 4326:
        xs, ys = (v[0] for v in rio_transform(src.crs, "EPSG:4326", [xs], [ys]))
    return float(ys), float(xs)       # lat, lon

rows, vecs, idx = [], [], 0
for tif, coords in tqdm(plan, desc="tiles"):
    if not coords: continue
    name = Path(tif).name
    try: src = rasterio.open(tif)
    except Exception: src = None
    with Image.open(tif) as im:
        im = im.convert("RGB")
        for s in range(0, len(coords), BATCH_SIZE):
            batch = coords[s:s + BATCH_SIZE]
            chips = [im.crop((x, y, x + CHIP_SIZE, y + CHIP_SIZE)) for x, y in batch]
            E = embed(chips)
            for (x, y), chip, e in zip(batch, chips, E):
                cid = f"{SOURCE}_{name}_s{CHIP_SIZE}_x{x}_y{y}"
                rel = f"thumbs/{cid}.webp"
                chip.resize((THUMB_SIZE, THUMB_SIZE)).save(OUT / rel, "WEBP", quality=80)
                lat, lon = latlon(src, x + CHIP_SIZE // 2, y + CHIP_SIZE // 2) if src else (None, None)
                rows.append(dict(idx=idx, chip_id=cid, lat=lat, lon=lon, scale=CHIP_SIZE, source=SOURCE,
                                 tile_id=name, x_offset=x, y_offset=y, chip_path=rel,
                                 ground_truth_class=chip_class(name, x, y)))
                vecs.append(e); idx += 1
    if src: src.close()

emb = np.stack(vecs).astype(np.float32)
meta = pd.DataFrame(rows)
print(emb.shape, "| mean norm:", float(np.linalg.norm(emb, axis=1).mean()))

## 5. Save

In [ ]:
np.save(OUT / "embeddings.npy", emb)
meta.to_parquet(OUT / "chips.parquet", index=False)
meta.to_csv(OUT / "chips.csv", index=False)
with zipfile.ZipFile(OUT / "thumbs.zip", "w", zipfile.ZIP_STORED) as z:
    for f in tqdm(sorted((OUT / "thumbs").iterdir()), desc="zip"): z.write(f, f"thumbs/{f.name}")
print(meta.head()); print(meta.ground_truth_class.notna().mean(), "of chips have a class label")
!du -sh {OUT}/*

## 6. Sanity check — nearest neighbours of a random chip

In [ ]:
import matplotlib.pyplot as plt
En = emb / np.linalg.norm(emb, axis=1, keepdims=True)
q = int(rng.integers(len(En)))
top = np.argsort(-(En @ En[q]))[:6]
fig, ax = plt.subplots(1, 6, figsize=(15, 3))
for a, i in zip(ax, top):
    a.imshow(Image.open(OUT / meta.chip_path[i])); a.axis("off")
    a.set_title(("query" if i == q else f"{float(En[i] @ En[q]):.2f}") + f"\n{meta.ground_truth_class[i]}", fontsize=8)
plt.show()

## 7. (Optional) Load into Supabase
Set `LOAD_TO_SUPABASE` / `UPLOAD_THUMBS` to `True` in the config cell and add the three secrets (names in the config cell) under Add-ons → Secrets. **Never paste the values into the notebook.**

Vectors stay raw/unnormalized, like the FAISS index. Rank with cosine distance (`<=>`); that gives the same ordering as the backend's inner-product search on a query vector.

In [ ]:
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ[name]

import psycopg
from pgvector.psycopg import register_vector
assert LOAD_TO_SUPABASE, 'LOAD_TO_SUPABASE is False — skipping (set True in the config cell)'

with psycopg.connect(secret(SECRET_DB_URL), autocommit=True) as conn:
    conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
    register_vector(conn)
    conn.execute("DROP TABLE IF EXISTS chips")
    conn.execute("""
        CREATE TABLE chips (
          idx INTEGER PRIMARY KEY, chip_id TEXT UNIQUE, lat DOUBLE PRECISION, lon DOUBLE PRECISION,
          scale INTEGER, source TEXT, tile_id TEXT, x_offset INTEGER, y_offset INTEGER,
          chip_path TEXT, ground_truth_class TEXT, embedding vector(384))""")
    cols = "idx,chip_id,lat,lon,scale,source,tile_id,x_offset,y_offset,chip_path,ground_truth_class"
    with conn.cursor() as cur, cur.copy(f"COPY chips ({cols},embedding) FROM STDIN") as cp:
        cp.set_types(["int4","text","float8","float8","int4","text","text","int4","int4","text","text","vector"])
        for r, e in tqdm(zip(meta[cols.split(",")].itertuples(index=False, name=None), emb), total=len(meta), desc="copy"):
            cp.write_row([None if (isinstance(v, float) and math.isnan(v)) else v for v in r] + [e])
    conn.execute("CREATE INDEX chips_hnsw ON chips USING hnsw (embedding vector_cosine_ops)")
    conn.execute("CREATE INDEX chips_class ON chips (ground_truth_class)")
    print(conn.execute("SELECT count(*) FROM chips").fetchone())

In [ ]:
assert LOAD_TO_SUPABASE
# Test query straight from SQL: 5 nearest chips to chip #0
with psycopg.connect(secret(SECRET_DB_URL)) as conn:
    register_vector(conn)
    for r in conn.execute("SELECT chip_id, 1-(embedding <=> %s) AS sim FROM chips ORDER BY embedding <=> %s LIMIT 5", (emb[0], emb[0])):
        print(r)

### Thumbnails → Supabase Storage
Create a **public** bucket named like `STORAGE_BUCKET` first (Storage → New bucket). Takes a few minutes with threads.

In [ ]:
assert UPLOAD_THUMBS, 'UPLOAD_THUMBS is False — skipping (set True in the config cell)'
from supabase import create_client
from concurrent.futures import ThreadPoolExecutor
sb = create_client(secret(SECRET_SB_URL), secret(SECRET_SB_KEY))
bucket = sb.storage.from_(STORAGE_BUCKET)
def up(p):
    try:
        bucket.upload(p.name, p.read_bytes(), {"content-type": "image/webp", "upsert": "true"}); return True
    except Exception as e: return str(e)
files = sorted((OUT / "thumbs").iterdir())
with ThreadPoolExecutor(16) as ex: res = list(tqdm(ex.map(up, files), total=len(files)))
print("uploaded:", sum(r is True for r in res), "| failed:", [r for r in res if r is not True][:3])
# public URL pattern: {SUPABASE_URL}/storage/v1/object/public/thumbs/<chip_id>.webp